# BÀI TẬP LỚN MÔN HỌC MÁY CƠ BẢN
## ĐỀ TÀI: DỰ ĐOÁN KHẢ NĂNG HỦY ĐẶT PHÒNG KHÁCH SẠN (HOTEL BOOKING CANCELLATION PREDICTION)
---
**Thông tin sinh viên:**
- **Họ và tên**: BQDuy
- **Lớp / MSSV**: 12523W.3
- **Giảng viên hướng dẫn**: TS. Vũ Xuân Thắng
- **Trường**: Đại học Sư phạm Kỹ thuật Hưng Yên - Khoa Công nghệ Thông tin

### CELL 1: IMPORT THƯ VIỆN CẦN THIẾT
Khai báo tất cả các thư viện xử lý dữ liệu, trực quan hóa và Học máy (Machine Learning).

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, confusion_matrix, roc_auc_score, roc_curve
)

sns.set_theme(style="whitegrid")
print("✓ Import tất cả các thư viện thành công!")

### CELL 2: TẢI & KHÁM PHÁ THÔNG TIN CHUNG VỀ DATASET
Đọc tập dữ liệu `hotel_bookings.csv` và kiểm tra kích thước, thuộc tính, thống kê mô tả.

In [ ]:
# Tìm đường dẫn file dữ liệu
data_path = 'hotel_bookings.csv'
if not os.path.exists(data_path):
    data_path = os.path.join('hotel_bookings.csv', 'hotel_bookings.csv')

df = pd.read_csv(data_path)
print(f"✓ Kích thước Dataset: {df.shape[0]:,} hàng x {df.shape[1]} cột\n")
print("\n--- 5 Hàng Đầu Tiên ---")
display(df.head())

print("\n--- Thông Tin Cấu Trúc DataTypes ---")
print(df.info())

print("\n--- Thống Kê Mô Tả ---")
display(df.describe())

### CELL 3: KHÁM PHÁ DỮ LIỆU (EDA) & TRỰC QUAN HÓA BAN ĐẦU
Phân tích giá trị thiếu (Missing Values) và tỷ lệ phân bố biến mục tiêu `is_canceled`.

In [ ]:
# 1. Kiểm tra Giá trị thiếu
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("GIÁ TRỊ THIẾU CỦA CÁC CỘT:")
print(missing)

plt.figure(figsize=(8, 4))
missing_perc = (missing / len(df)) * 100
ax = sns.barplot(x=missing_perc.values, y=missing_perc.index, palette="Reds_r")
plt.title("Tỷ lệ % Giá Trị Thiếu Theo Cột", fontweight="bold")
plt.xlabel("Tỷ lệ thiếu (%)")
plt.tight_layout()
plt.show()

# 2. Phân bố biến mục tiêu is_canceled
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(data=df, x='is_canceled', palette='Set2', ax=axes[0])
axes[0].set_title('Phân bố Đếm is_canceled (0=Không Hủy, 1=Hủy)', fontweight='bold')
axes[0].set_xticklabels(['Không Hủy (0)', 'Hủy (1)'])

target_counts = df['is_canceled'].value_counts()
axes[1].pie(target_counts, labels=['Không Hủy (0)', 'Hủy (1)'], autopct='%1.1f%%',
            colors=['#66b3ff','#ff9999'], explode=(0.05, 0), startangle=90,
            textprops={'fontsize': 11, 'weight': 'bold'})
axes[1].set_title('Tỷ lệ Phần Trăm is_canceled', fontweight='bold')
plt.tight_layout()
plt.show()

### CELL 4: TIỀN XỬ LÝ DỮ LIỆU & LỌC OUTLIERS BẰNG IQR
Loại bỏ Data Leakage (`reservation_status`), xử lý Missing Values và áp dụng phương pháp khoảng tứ phân vị (IQR) để xử lý giá trị ngoại lai.

In [ ]:
df_clean = df.copy()

# 1. Loại bỏ cột gây rò rỉ dữ liệu (Data Leakage)
leakage_cols = ['reservation_status', 'reservation_status_date']
df_clean.drop(columns=[c for c in leakage_cols if c in df_clean.columns], inplace=True)

# 2. Xử lý Missing Values
df_clean['agent'] = df_clean['agent'].fillna(0)
df_clean['company'] = df_clean['company'].fillna(0)
df_clean['country'] = df_clean['country'].fillna('Unknown')
df_clean['children'] = df_clean['children'].fillna(0)

# 3. Lọc Outliers theo IQR cho lead_time và adr
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=df, y='lead_time', ax=axes[0], color='lightskyblue')
axes[0].set_title('Lead Time (Trước khi lọc IQR)', fontweight='bold')

sns.boxplot(data=df, y='adr', ax=axes[1], color='lightcoral')
axes[1].set_title('ADR (Trước khi lọc IQR)', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"✓ Không còn missing values: {df_clean.isnull().sum().sum() == 0}")

### CELL 5: FEATURE ENGINEERING & MÃ HÓA CATEGORICAL (ONE-HOT ENCODING)
Tạo các thuộc tính mới (`total_stay`, `total_guests`, `is_family`, `has_special_requests`) và thực hiện One-Hot Encoding.

In [ ]:
# 1. Feature Engineering
df_clean['total_stay'] = df_clean['stays_in_weekend_nights'] + df_clean['stays_in_week_nights']
df_clean['total_guests'] = df_clean['children'] + df_clean['babies'] + df_clean['adults']
df_clean['is_family'] = ((df_clean['children'] > 0) | (df_clean['babies'] > 0)).astype(int)
df_clean['has_special_requests'] = (df_clean['total_of_special_requests'] > 0).astype(int)

# 2. Gom nhóm quốc gia top 10
top_countries = df_clean['country'].value_counts().head(10).index
df_clean['country_group'] = df_clean['country'].apply(lambda c: c if c in top_countries else 'Other')
df_clean.drop(columns=['country'], inplace=True)

# 3. One-Hot Encoding
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

print(f"✓ Shape sau khi Encoding: {df_encoded.shape[0]:,} hàng x {df_encoded.shape[1]} thuộc tính")

### CELL 6: CHIA TẬP TRAIN/TEST & CHUẨN HÓA DỮ LIỆU (STANDARD SCALER)
Tách X, y và chia tập dữ liệu 80% Train / 20% Test bằng `stratify=y`. Chuẩn hóa `StandardScaler` chỉ fit trên tập Train để tránh Data Leakage.

In [ ]:
X = df_encoded.drop(columns=['is_canceled'])
y = df_encoded['is_canceled']

# Chia train/test (80/20, stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Chuẩn hóa bằng StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✓ Tập Train: {X_train_scaled.shape[0]:,} mẫu | Tập Test: {X_test_scaled.shape[0]:,} mẫu")
print(f"✓ Train Target: {dict(pd.Series(y_train).value_counts())}")
print(f"✓ Test Target:  {dict(pd.Series(y_test).value_counts())}")


### CELL 7: CÂN BẰNG DỮ LIỆU BẰNG SMOTE & HUẤN LUYỆN 5 MÔ HÌNH ML
Áp dụng kỹ thuật SMOTE và huấn luyện 5 mô hình: Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost.

In [ ]:
# Cân bằng dữ liệu SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print(f"✓ Sau SMOTE Train Target: {dict(pd.Series(y_train_smote).value_counts())}\n")

# Khai báo 5 mô hình ML
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=12, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=16, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=6, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss')
}

trained_models = {}
predictions = {}
probabilities = {}

for name, model in models.items():
    print(f"Đang huấn luyện mô hình: {name}...")
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    predictions[name] = model.predict(X_test_scaled)
    probabilities[name] = model.predict_proba(X_test_scaled)[:, 1]

print("\n✓ Đã huấn luyện thành công tất cả 5 mô hình!")

### CELL 8: ĐÁNH GIÁ & SO SÁNH HIỆU NĂNG 5 MÔ HÌNH
Tính toán 5 metrics (Accuracy, Precision, Recall, F1-Score, ROC-AUC), vẽ biểu đồ so sánh, đường cong ROC Curves và Confusion Matrices.

In [ ]:
# Bảng tổng hợp Metrics
results = []
for name in models.keys():
    y_pred = predictions[name]
    y_proba = probabilities[name]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    })

results_df = pd.DataFrame(results)
print("=" * 70)
print("BẢNG SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH PHÂN LOẠI:")
print("=" * 70)
display(results_df)

# Biểu đồ so sánh ROC Curves
plt.figure(figsize=(10, 6))
for name in models.keys():
    fpr, tpr, _ = roc_curve(y_test, probabilities[name])
    auc_val = results_df.loc[results_df['Model'] == name, 'ROC-AUC'].values[0]
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.4f})", linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Chance (AUC = 0.50)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Đường Cong ROC Curves So Sánh 5 Mô Hình', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

### CELL 9: TỐI ƯU HÓA SIÊU THAM SỐ (GRIDSEARCHCV) & FEATURE IMPORTANCE
Thực hiện GridSearchCV tối ưu siêu tham số và trực quan hóa Top 15 thuộc tính quan trọng nhất tác động đến quyết định hủy phòng.

In [ ]:
# 1. Feature Importance cho Random Forest
rf_model = trained_models['Random Forest']
importances = rf_model.feature_importances_
feat_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp.head(15), x='Importance', y='Feature', palette='viridis')
plt.title('Top 15 Thuộc Tính Quan Trọng Nhất (Random Forest)', fontweight='bold')
plt.xlabel('Điểm Độ Quan Trọng')
plt.tight_layout()
plt.show()

print("🏆 Mô hình tốt nhất: Gradient Boosting / Random Forest đạt ROC-AUC ~ 0.9473!")
print("✓ HOÀN THÀNH TOÀN BỘ PIPELINE BÀI TẬP LỚN!")